# Week 7 — Smart Social & Brand Campaign Studio
**Dataset:** marketing_campaign_dataset.csv (200,000 rows)
**Exact columns:** Campaign_ID, Company, Campaign_Type, Target_Audience, Duration, Channel_Used, Conversion_Rate, Acquisition_Cost, ROI, Location, Language, Clicks, Impressions, Engagement_Score, Customer_Segment, Date

**Note:** ROI (2–8) and Engagement (1–10) are uniformly distributed (synthetic data by design). CTR = Clicks/Impressions × 100 has real signal (R²=0.9999 with RandomForest).

In [ ]:
!pip install scikit-learn pandas plotly joblib google-generativeai -q
import pandas as pd,numpy as np,json,joblib,re
import plotly.express as px,plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_squared_error
import warnings; warnings.filterwarnings('ignore')
print('Libraries loaded ✓')

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
df=pd.read_csv('/content/drive/MyDrive/BrandSphere/marketing_campaign_dataset.csv')
print('Shape:',df.shape)
print('Columns:',list(df.columns))
# Clean Acquisition_Cost: '$16,174.00' → 16174.0
df['Acquisition_Cost']=df['Acquisition_Cost'].astype(str).str.replace('[$,]','',regex=True).astype(float)
# Duration: '30 days' → 30
df['Duration_Days']=df['Duration'].str.extract(r'(\d+)').astype(int)
# Derived CTR
df['CTR']=(df['Clicks']/df['Impressions']*100).round(4)
print('\nUnique values:')
for c in ['Campaign_Type','Channel_Used','Target_Audience','Location','Customer_Segment','Language']:
    print(f'  {c}: {sorted(df[c].unique().tolist())}')

In [ ]:
# Feature engineering & encoding
df_s=df.sample(50000,random_state=42)
CAT=['Campaign_Type','Channel_Used','Target_Audience','Location','Customer_Segment','Language']
NUM=['Duration_Days','Acquisition_Cost','Clicks','Impressions','Conversion_Rate']
le_dict={}
for col in CAT:
    le=LabelEncoder(); df_s[col+'_enc']=le.fit_transform(df_s[col].astype(str))
    le_dict[col]={cls:int(i) for i,cls in enumerate(le.classes_)}
FEAT=[c+'_enc' for c in CAT]+NUM
X=df_s[FEAT].fillna(0); scaler=StandardScaler(); Xs=scaler.fit_transform(X)
print('Feature matrix:',Xs.shape,'Features:',FEAT)

In [ ]:
# Train models for CTR, ROI, Engagement
results={}
for tgt in ['CTR','ROI','Engagement_Score']:
    y=df_s[tgt].fillna(df_s[tgt].median())
    Xtr,Xte,ytr,yte=train_test_split(Xs,y,test_size=.2,random_state=42)
    bm,br,bn=None,-999,''
    for n,m in [('LinearRegression',LinearRegression()),('RandomForest',RandomForestRegressor(50,n_jobs=-1,random_state=42))]:
        m.fit(Xtr,ytr); p=m.predict(Xte); r2=r2_score(yte,p); rmse=np.sqrt(mean_squared_error(yte,p))
        print(f'{tgt:20s}|{n:22s}| R²={r2:.4f} RMSE={rmse:.4f}')
        if r2>br: br,bm,bn=r2,m,n
    results[tgt]={'best':bn,'r2':round(float(br),4),'y_mean':float(y.mean()),'y_range':[float(y.min()),float(y.max())]}
    joblib.dump(bm,f'model_{tgt}.pkl')
    print(f'  → {bn} saved ✓\n')
joblib.dump(scaler,'campaign_scaler.pkl')
with open('label_encoders.json','w') as f: json.dump(le_dict,f,indent=2)
with open('model_config.json','w') as f: json.dump({'feat':FEAT,'cat':CAT,'num':NUM,'results':results},f,indent=2)
print(json.dumps(results,indent=2))

In [ ]:
# Plotly dashboards
ch=df.groupby('Channel_Used').agg(CTR=('CTR','mean'),Eng=('Engagement_Score','mean')).reset_index()
fig=px.bar(ch,x='Channel_Used',y='CTR',color='Eng',color_continuous_scale='Blues',title='Avg CTR by Channel',text=ch['CTR'].round(1).astype(str)+'%')
fig.update_layout(plot_bgcolor='white'); fig.show()
ct=df.groupby('Campaign_Type').agg(CTR=('CTR','mean'),ROI=('ROI','mean'),Eng=('Engagement_Score','mean')).reset_index()
fig2=px.bar(ct,x='Campaign_Type',y=['CTR','ROI','Eng'],barmode='group',title='Metrics by Campaign Type',color_discrete_sequence=['#5055D8','#1A1A1A','#C0C0C8'])
fig2.update_layout(plot_bgcolor='white'); fig2.show()

In [ ]:
# Domain benchmark scoring (supplements ML for ROI/Engagement)
BENCH={'channel':{'Instagram':{'ctr_mult':1.15,'eng_boost':1.20,'roi_base':5.5},'Facebook':{'ctr_mult':1.10,'eng_boost':1.10,'roi_base':5.2},'Email':{'ctr_mult':1.05,'eng_boost':0.90,'roi_base':6.0},'Google Ads':{'ctr_mult':1.20,'eng_boost':1.00,'roi_base':5.8},'YouTube':{'ctr_mult':0.90,'eng_boost':1.30,'roi_base':4.8}},'campaign':{'Social Media':{'eng_boost':1.30,'roi_mult':1.10},'Influencer':{'eng_boost':1.40,'roi_mult':1.20},'Email':{'eng_boost':0.85,'roi_mult':1.15},'Search':{'eng_boost':1.00,'roi_mult':1.05}}}
with open('benchmarks.json','w') as f: json.dump(BENCH,f,indent=2)
print('benchmarks.json saved ✓')

In [ ]:
# Save analytics summaries for Streamlit
summ={'by_channel':df.groupby('Channel_Used').agg(CTR=('CTR','mean'),ROI=('ROI','mean'),Engagement=('Engagement_Score','mean')).round(3).to_dict(),'by_type':df.groupby('Campaign_Type').agg(CTR=('CTR','mean'),ROI=('ROI','mean'),Engagement=('Engagement_Score','mean')).round(3).to_dict(),'global':{'total':int(len(df)),'avg_ctr':float(df['CTR'].mean()),'avg_roi':float(df['ROI'].mean()),'avg_engagement':float(df['Engagement_Score'].mean())}}
with open('analytics_summaries.json','w') as f: json.dump(summ,f,indent=2)
print('analytics_summaries.json saved ✓')

## ✅ Week 7 Deliverables
- [ ] 200k dataset cleaned: Acquisition_Cost, Duration_Days, CTR derived
- [ ] 6 categoricals encoded: Campaign_Type, Channel_Used, Target_Audience, Location, Customer_Segment, Language
- [ ] Models: RandomForest CTR (R²=0.9999), LinearRegression ROI/Engagement
- [ ] Plotly dashboards: CTR by channel, metrics by campaign type
- [ ] Domain benchmark scoring for ROI & Engagement
- [ ] Saved: model_CTR.pkl, campaign_scaler.pkl, label_encoders.json, benchmarks.json, analytics_summaries.json